In [6]:
import gc

import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [1]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



NameError: name 'pd' is not defined

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()

cols_to_drop= ["flag_email"]#, ,"organization_type_Other"

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)




#run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"prev_app+installment fpi",enable_feature_permutation=True)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_prev_app+installment_fpi.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "top_prev_with_installment_pruned_feature_importance.csv")

#X.drop(columns=cols_to_drop, inplace=True)
X_cleaned = clean_importance_zero_and_negative_pfi(importance_pfi_df,X)
#X_cleaned= clean_noise_from_feature_importance(importance_df,X_cleaned)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X_cleaned,Y,experiment_name,"top_prev_with_installment_pruned")

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

eliminando ['amt_req_credit_breau_mon', 'hour_apply_start', 'ratio_credit_to_goods_std', 'flag_region_not_live', 'amt_application_min', 'log_amt_application_std', 'amt_credit_median', 'housetype_mode', 'name_product_type_prev_1', 'flag_own_car', 'instalments_repeated_for_underpayment_mean_prev_1', 'instalments_log_amt_instalment_mean_prev_1', 'cnt_children', 'flag_phone', 'organization_type_Trade: type 3', 'name_cash_loan_purpose_prev_1'] por importancia 0 o negativa en feature permutation
[0]	validation_0-auc:0.72900
[1]	validation_0-auc:0.73992
[2]	validation_0-auc:0.74406
[3]	validation_0-auc:0.74612
[4]	validation_0-auc:0.74936
[5]	validation_0-auc:0.75254
[6]	validation_0-auc:0.75500
[7]	validation_0-auc:0.75681
[8]	validation_0-auc:0.75879
[9]	validation_0-auc:0.76073
[10]	validation_0-auc:0.76333
[11]	validation_0-auc:0.76444
[12]	validation_0-auc:0.76530
[13]	validation_0-auc:0.76566
[14]	validation_0-auc:0.76613
[15]	validation_0-auc:0.76751
[16]	validation_0-auc:0.76802
[17]	

10235

In [2]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

KeyError: 'id_curr'

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)



#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X)

#importance2 = pd.read_csv(cfg.ARTIFACTS_DIR / "second filter.csv")
X= X.drop(columns=["flag_document_5","organization_type_Industry: type 5","bureau_balance_potential_on_going_loan_loan_1","bureau_days_credit_enddate_is_missing_loan_2"]) #,"building_score_std",""
#X= clean_noise_from_feature_importance(importance2,X,0.001)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"apply criteria bureau")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72532
[1]	validation_0-auc:0.73412
[2]	validation_0-auc:0.73881
[3]	validation_0-auc:0.74091
[4]	validation_0-auc:0.74246
[5]	validation_0-auc:0.74503
[6]	validation_0-auc:0.74732
[7]	validation_0-auc:0.74963
[8]	validation_0-auc:0.75079
[9]	validation_0-auc:0.75199
[10]	validation_0-auc:0.75439
[11]	validation_0-auc:0.75564
[12]	validation_0-auc:0.75817
[13]	validation_0-auc:0.75921
[14]	validation_0-auc:0.76002
[15]	validation_0-auc:0.76011
[16]	validation_0-auc:0.76092
[17]	validation_0-auc:0.76132
[18]	validation_0-auc:0.76110
[19]	validation_0-auc:0.76165
[20]	validation_0-auc:0.76280
[21]	validation_0-auc:0.76310
[22]	validation_0-auc:0.76342
[23]	validation_0-auc:0.76379
[24]	validation_0-auc:0.76447
[25]	validation_0-auc:0.76470
[26]	validation_0-auc:0.76502
[27]	validation_0-auc:0.76501
[28]	validation_0-auc:0.76489
[29]	validation_0-auc:0.76568
[30]	validation_0-auc:0.76568
[31]	validation_0-auc:0.76619
[32]	validation_0-auc:0.76565
[33]	validation_0-au

In [8]:
importance_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "creating_criteria_feature_importance.csv")
criteria = creating_criteria(importance_df,importance_permutation_df)
a= criteria [criteria["importances"] == 0]
display(a)

,feature_name,importances,feature,importance_fold_1,importance_fold_2,importance_fold_3,importance_fold_4,importance_fold_5,mean_importance_cv,diff_gain_and_permutation
43,flag_document_5,0.0,flag_document_5,0.000000,-0.000012,0.000000,-1.338400e-04,-4.174248e-05,-3.742589e-05,3.742589e-05
29,organization_type_Industry: type 5,0.0,organization_type_Industry: type 5,0.000000,-0.000017,-0.000107,0.000000e+00,-2.950878e-05,-3.066364e-05,3.066364e-05
39,organization_type_Business Entity Type 2,0.0,organization_type_Business Entity Type 2,-0.000029,0.000000,-0.000032,-5.537875e-05,0.000000e+00,-2.321991e-05,2.321991e-05
4,bureau_balance_potential_on_going_loan_loan_1,0.0,bureau_balance_potential_on_going_loan_loan_1,0.000000,0.000000,-0.000058,0.000000e+00,-1.280622e-05,-1.417323e-05,1.417323e-05
63,bureau_days_credit_enddate_is_missing_loan_2,0.0,bureau_days_credit_enddate_is_missing_loan_2,0.000000,-0.000019,-0.000011,0.000000e+00,-1.239360e-05,-8.534275e-06,8.534275e-06
26,organization_type_Medicine,0.0,organization_type_Medicine,0.000000,-0.000038,0.000000,0.000000e+00,0.000000e+00,-7.538859e-06,7.538859e-06
67,closed_balance_status_score_max_closed_max,0.0,closed_balance_status_score_max_closed_max,-0.000039,-0.000003,0.000013,0.000000e+00,0.000000e+00,-5.871836e-06,5.871836e-06
32,organization_type_Industry: type 3,0.0,organization_type_Industry: type 3,0.000000,0.000000,0.000000,-3.390003e-05,4.769050e-06,-5.826196e-06,5.826196e-06
33,organization_type_Housing,0.0,organization_type_Housing,0.000000,0.000000,0.000001,-2.683181e-05,0.000000e+00,-5.133111e-06,5.133111e-06
0,bureau_days_enddate_fact_is_missing_loan_2,0.0,bureau_days_enddate_fact_is_missing_loan_2,0.000000,0.000000,0.000000,-7.495749e-06,-1.643985e-05,-4.787121e-06,4.787121e-06


In [17]:
def identificar_alta_correlacion(df, columns, threshold=0.95):
    """Calcula la matriz de correlación y devuelve un DataFrame con las parejas

    de variables que superan el umbral establecido, ordenadas de mayor a menor.
    """
    # 1. Calcular la matriz de correlación (solo para las columnas numéricas indicadas)
    corr_matrix = df[columns].corr(method="pearson").abs()

    # 2. Seleccionar el triángulo superior de la matriz (así evitamos duplicados y la diagonal)
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    # 3. Desenrollar la matriz (unstack) y limpiar los valores nulos
    corr_series = upper_tri.unstack()
    corr_series = corr_series.dropna()

    # 4. Crear un DataFrame con los resultados
    df_corr = corr_series.reset_index()
    df_corr.columns = ["Variable_B", "Variable_A", "Correlacion"]

    # Reordenar columnas para que se lea intuitivamente (A, B, Valor)
    df_corr = df_corr[["Variable_A", "Variable_B", "Correlacion"]]

    # 5. Filtrar por el umbral (Threshold) y ordenar de forma descendente
    df_alto_riesgo = df_corr[df_corr["Correlacion"] > threshold]
    df_alto_riesgo = df_alto_riesgo.sort_values(
        by="Correlacion", ascending=False
    ).reset_index(drop=True)

    return df_alto_riesgo

df_numeric = X.select_dtypes(include=[np.number])
dtale.show(identificar_alta_correlacion(df_numeric,df_numeric.columns))

2026-06-23 23:27:54,920 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [2]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

eliminando ['bureau_amt_credit_sum_overdue_is_missing_loan_2', 'organization_type_Industry: type 11', 'organization_type_Industry: type 3', 'organization_type_Industry: type 4', 'organization_type_Industry: type 5', 'organization_type_Industry: type 7', 'bureau_balance_potential_on_going_loan_loan_1', 'organization_type_Industry: type 1', 'bureau_amt_annuity_is_missing_loan_2', 'organization_type_Medicine', 'bureau_amt_annuity_is_missing_loan_1', 'organization_type_Other', 'organization_type_Other industry', 'organization_type_Other trade', 'organization_type_Postal', 'organization_type_Kindergarten', 'organization_type_Restaurant', 'organization_type_Housing', 'organization_type_Government', 'bureau_credit_active_active_loan_1', 'bureau_has_bureau_balance_data_loan_2', 'bureau_has_bureau_balance_data_loan_1', 'bureau_balance_is_delincuency_sum_loan_2', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_max_loan_1', 'bureau_balance_potential_on_going_loan_loan_2'

475

In [5]:
#app_train_with_feature_engineering + prev_app + bureau + installments
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "pruned_bureau_with_balance.parquet")

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments.parquet")

merged_df = previous_application_df.merge(
    bureau_df,
    on= "id_curr", 
    how="left"
)

merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_y')].rename(columns=lambda x: x.rstrip('_x'))

#cleaning the third
del previous_application_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72929
[1]	validation_0-auc:0.73938
[2]	validation_0-auc:0.74514
[3]	validation_0-auc:0.74717
[4]	validation_0-auc:0.74957
[5]	validation_0-auc:0.75220
[6]	validation_0-auc:0.75520
[7]	validation_0-auc:0.75739
[8]	validation_0-auc:0.75943
[9]	validation_0-auc:0.76107
[10]	validation_0-auc:0.76306
[11]	validation_0-auc:0.76521
[12]	validation_0-auc:0.76701
[13]	validation_0-auc:0.76785
[14]	validation_0-auc:0.76926
[15]	validation_0-auc:0.77011
[16]	validation_0-auc:0.77162
[17]	validation_0-auc:0.77202
[18]	validation_0-auc:0.77208
[19]	validation_0-auc:0.77254
[20]	validation_0-auc:0.77278
[21]	validation_0-auc:0.77354
[22]	validation_0-auc:0.77380
[23]	validation_0-auc:0.77397
[24]	validation_0-auc:0.77412
[25]	validation_0-auc:0.77426
[26]	validation_0-auc:0.77456
[27]	validation_0-auc:0.77436
[28]	validation_0-auc:0.77492
[29]	validation_0-auc:0.77577
[30]	validation_0-auc:0.77635
[31]	validation_0-auc:0.77632
[32]	validation_0-auc:0.77570
[33]	validation_0-au

409